In [ ]:
import pandas as pd
import numpy as np
import torch
import ast
from collections import Counter
import argparse
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from codecarbon import EmissionsTracker
import matplotlib.pyplot as plt

# Initialize tracker (logs to a CSV file)
tracker = EmissionsTracker(output_dir=".", log_level="error")  # Suppress verbose logs

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Your existing landmark_df loaded
# landmark_df = pd.read_csv('landmark_df.csv')
landmark_df = pd.read_csv('landmark_df_evo.csv', na_values=['', 'None', 'NaN', 'na', 'nan'])
landmark_df = landmark_df.fillna('')

max_visits = 3  # Example for patients with exactly 5 visits
visit_counts = landmark_df['subject_id'].value_counts()
selected_patients = visit_counts[visit_counts == max_visits].index

df_selected = landmark_df[landmark_df['subject_id'].isin(selected_patients)].copy()

results = []
SEED = 42

def old_narrative_prompt(row):
    narrative = f"Patient is a {row['age_at_landmark']}-year-old {row['gender']}."
    narrative += f" This is the {row['num_total_visits']} visit."
    if row['days_since_previous_visit'] != -1:
        narrative += f" The last visit happened {row['days_since_previous_visit']} days ago."

    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" Medical history includes: {row['diag_text']}."

    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" Current medications are: {row['med_text']}."

    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" Procedures performed: {row['proc_text']}."
    
    narrative += "Given that Probabilities > 0.5 indicate a higher risk of mortality, while probabilities < 0.5 indicate a lower risk. \n \
            Based on this information, what is the probability of mortality within 90 days?"

    return narrative
def narrative_prompt(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {', '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {', '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {', '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {', '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {', '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {', '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative

class ClinicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
try:
    tracker.start()
    
    # Your code here (e.g., training an ML model)
    # ------------------------------

    # Store predictions and labels for confidence intervals
    predictions_list = []
    labels_list = []

    for landmark_visit in range(1, max_visits + 1):
        print(f"Fine-tuning and evaluating at Landmark {landmark_visit}")

        df_subset = df_selected[df_selected['landmark_visit'] == landmark_visit]

        patients = df_subset['subject_id'].unique()

        train_patients, test_patients = train_test_split(
            patients,
            test_size=0.2,
            random_state=SEED,
            stratify=df_subset.groupby('subject_id')['death_in_90days'].max()
        )

        train_df = df_subset[df_subset['subject_id'].isin(train_patients)].copy()
        test_df = df_subset[df_subset['subject_id'].isin(test_patients)].copy()
        
        # Clearly apply the narrative_prompt function
        train_texts = train_df.apply(narrative_prompt, axis=1).tolist()
        train_labels = train_df['death_in_90days'].tolist()

        test_texts = test_df.apply(narrative_prompt, axis=1).tolist()
        test_labels = test_df['death_in_90days'].tolist()

        # train_texts = train_df.apply(lambda row: ' '.join(filter(None, [row['med_text'], row['diag_text'], row['proc_text']])), axis=1).tolist()
        # train_labels = train_df['death_in_90days'].tolist()

        # test_texts = test_df.apply(lambda row: ' '.join(filter(None, [row['med_text'], row['diag_text'], row['proc_text']])), axis=1).tolist()
        # test_labels = test_df['death_in_90days'].tolist()

        train_dataset = ClinicalDataset(train_texts, train_labels, tokenizer)
        test_dataset = ClinicalDataset(test_texts, test_labels, tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        
        # model.train()
        # for epoch in range(epochs):
        #     total_loss = 0
        #     for batch in train_loader:
        #         optimizer.zero_grad()
        #         inputs = {k: v.to(device) for k, v in batch.items()}
        #         outputs = model(**inputs)
        #         loss = outputs.loss
        #         loss.backward()
        #         optimizer.step()
        #         total_loss += loss.item()
        #     avg_loss = total_loss / len(train_loader)
        #     print(f'Landmark {landmark_visit}, Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')

        epochs = 5  # Allow more epochs
        patience = 1  # Stop if no improvement after 3 epochs
        best_val_auc = 0.0
        epochs_no_improve = 0

        for epoch in range(epochs):
            model.train()
            total_train_loss = 0

            # Training phase
            for batch in train_loader:
                optimizer.zero_grad()
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            avg_train_loss = total_train_loss / len(train_loader)

            # Validation phase
            model.eval()
            val_predictions, val_labels = [], []

            with torch.no_grad():
                total_val_loss = 0
                for batch in test_loader:
                    inputs = {k: v.to(device) for k, v in batch.items()}
                    outputs = model(**inputs)
                    val_loss = outputs.loss
                    total_val_loss += val_loss.item()
                    
                    probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
                    val_predictions.extend(probs)
                    val_labels.extend(batch['labels'].cpu().numpy())

            avg_val_loss = total_val_loss / len(test_loader)
            val_auc = roc_auc_score(val_labels, val_predictions)
            f1 = f1_score(val_labels, (np.array(val_predictions) > 0.5).astype(int), zero_division=0)

            print(f'Epoch {epoch+1}/{epochs} | '
                f'Train Loss: {avg_train_loss:.4f} | '
                f'Validation Loss: {avg_val_loss:.4f} | '
                f'Validation AUC: {val_auc:.4f} | '
                f'F1 Score: {f1:.4f}')

            # Check early stopping conditions
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                # Optionally save best model weights:
                torch.save(model.state_dict(), 'best_model.pt')
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f'Early stopping triggered after epoch {epoch+1}')
                    break

        # Load best model after early stopping:
        model.load_state_dict(torch.load('best_model.pt'))
        
        model.eval()
        predictions, true_labels = [], []

        with torch.no_grad():
            for batch in test_loader:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
                predictions.extend(probs)
                true_labels.extend(batch['labels'].cpu().numpy())

        auc = roc_auc_score(true_labels, predictions)
        mortality_rate = np.mean(test_labels) * 100
        num_patients = len(test_df)

        results.append({
            'Landmark (Visit Number)': landmark_visit,
            'Number of Patients (Test)': num_patients,
            'Mortality (%)': f"{mortality_rate:.1f}%",
            'AUC MedBERT': f"{auc:.4f}"
        })

        predictions_list.append(predictions)
        labels_list.append(true_labels)
    
    # ------------------------------
    
finally:
    emissions: float = tracker.stop()  # Stops tracking and returns emissions in kgCO₂eq

print(f"Estimated carbon footprint: {emissions} kg CO₂eq")

In [ ]:
# Confidence interval calculation
def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.95):
    bootstrapped_scores = []
    rng = np.random.RandomState(SEED)
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(np.array(y_true)[indices])) < 2:
            continue
        score = roc_auc_score(np.array(y_true)[indices], np.array(y_pred)[indices])
        bootstrapped_scores.append(score)

    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    lower = sorted_scores[int((1.0 - alpha) / 2 * len(sorted_scores))]
    upper = sorted_scores[int((alpha + (1.0 - alpha) / 2) * len(sorted_scores))]
    return lower, upper

landmark_visits = list(range(1, max_visits + 1))
auc_means = []
ci_lowers = []
ci_uppers = []

for preds, labels in zip(predictions_list, labels_list):
    auc = roc_auc_score(labels, preds)
    ci_lower, ci_upper = bootstrap_auc_ci(labels, preds)
    auc_means.append(auc)
    ci_lowers.append(ci_lower)
    ci_uppers.append(ci_upper)

# Plot AUC with CI
plt.figure(figsize=(10, 6))
plt.plot(landmark_visits, auc_means, marker='o', color='steelblue', label='Mean AUC')
plt.fill_between(landmark_visits, ci_lowers, ci_uppers, color='steelblue', alpha=0.2, label='95% CI')
plt.xticks(landmark_visits)
plt.xlabel('Landmark Visit')
plt.ylabel('AUC')
plt.title('MedBERT Predictive Performance by Landmark Visit')
plt.legend()
plt.grid(alpha=0.4)
plt.ylim([0.5, 1.0])
plt.show()

results_df = pd.DataFrame(results)
print(results_df)
